<a href="https://colab.research.google.com/github/Gregfili/DZ8_Classes/blob/master/Embedding_n_LMM_ver_3_(%D0%93%D1%80%D0%B8%D0%B3%D0%BE%D1%80%D0%B8%D0%B92).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом эксперементе я поменял модель и веса. Я использовал sentence-transformers/distilbert-multilingual-nli-stsb-quora-ranking"  и веса "тематика": 0.4,
    "аудитория": 0.35,
    "качество контента": 0.05,
    "активность": 0.15,
    "соответствие формату": 0.05

Этот код предназначен для автоматического анализа и ранжирования Telegram-каналов на основе их соответствия заданным креативным текстам (например, рекламным объявлениям или описаниям проектов). Он использует методы обработки естественного языка (NLP) и искусственного интеллекта (ИИ) для оценки релевантности каналов и предоставляет подробный отчет с результатами.

Основные задачи, выполняемые кодом:

1. Извлечение данных из базы данных SQLite: Код подключается к базе данных, содержащей информацию о Telegram-каналах (название, описание, категория, подписчики, сообщения) и извлекает необходимые данные для анализа.

2. Очистка и предобработка текста: Код очищает тексты сообщений и описаний каналов от шума (ссылки, эмодзи, специальные символы), приводя их к стандартному виду для дальнейшей обработки.

3. Векторное представление текстов (эмбеддинги): Используя мощные модели Sentence Transformers (HuggingFace), код преобразует текстовые описания каналов и креативные тексты в числовые векторы (эмбеддинги). Это позволяет сравнивать тексты на основе их смыслового сходства.

4. Поиск похожих каналов: С помощью алгоритма FAISS (Facebook AI Similarity Search) код быстро находит каналы, чьи векторные представления наиболее близки к векторным представлениям креативных текстов. Это позволяет отобрать каналы, тематически связанные с заданными креативами.

5. Анализ активности каналов: Код анализирует активность каналов (просмотры, репосты, реакции) за последние 20 сообщений и рассчитывает оценку активности по 10-балльной шкале. Это позволяет оценить, насколько "живой" канал и как часто пользователи взаимодействуют с его контентом.

6. Ранжирование каналов с помощью Mistral-Nemo: Используя большую языковую модель Mistral-Nemo, код оценивает каждый канал по нескольким критериям: тематика, аудитория, качество контента, активность, соответствие формату. Mistral-Nemo предоставляет обоснование для каждой оценки и выставляет итоговый рейтинг каналу, учитывая заданные веса критериев. Этот этап позволяет получить более глубокую и комплексную оценку релевантности канала, чем простое сравнение эмбеддингов.

7. Сохранение результатов в Google Sheets: Код автоматически создает Google Таблицу и записывает в нее результаты анализа: название канала, описание, категорию, оценку активности, рейтинг Mistral-Nemo, комментарии Mistral-Nemo и другие данные. Это позволяет удобно просматривать и анализировать результаты, а также делиться ими с другими пользователями. Авторизация в Google Sheets происходит с помощью встроенной в Google Colab системы.

В итоге, код предоставляет инструмент для автоматического подбора Telegram-каналов для размещения рекламы или продвижения проектов, основываясь на смысловом анализе текстов, оценке активности и экспертной оценке Mistral-Nemo. Это значительно упрощает и ускоряет процесс выбора подходящих площадок для продвижения.

In [1]:
!pip install jedi cmake setuptools wheel scikit-build cmake ninja
!pip install faiss-cpu
!pip install langchain
!pip install langchain_community
!pip install langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.9/422.9 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.21
    Uninstalling langchain-core-0.3.21:
      Successfully uninstalled langchain-core-0.3.21
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.9
    Uninstalling langchain-0.3.9:
      Successfully uninstalled langchain-0.3.9


In [2]:
# Настройка API Google Sheets
import gspread
from google.colab import auth
from google.auth import default

# Аутентификация в Google Colab
auth.authenticate_user()
creds, _ = default()
client = gspread.authorize(creds)


Установка, если происходит сбой и перезапуск среды запустить выполнение ячейки заново!!! Так как происходит компилляция библиотеки установка довольно долгая



In [3]:
import os
!pip cache purge

# Настройка переменных окружения для поддержки CUDA
os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75"
os.environ["FORCE_CMAKE"] = "1"

# Установка библиотеки!pip uninstall -y llama-cpp-python
!pip uninstall -y llama-cpp-python
!pip install llama-cpp-python --force-reinstall --no-cache-dir --verbose

!pip uninstall -y numpy
!pip install "numpy<2.0.0,>=1.22.4"
!pip check

Files removed: 90
Using pip 24.1.2 from /usr/local/lib/python3.10/dist-packages/pip (python 3.10)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 MB 139.0 MB/s eta 0:00:00
  Running command pip subprocess to install build dependencies
  Using pip 24.1.2 from /usr/local/lib/python3.10/dist-packages/pip (python 3.10)
  Non-user install by explicit request
  Created build tracker: /tmp/pip-build-tracker-t1gz1yt6
  Entered build tracker: /tmp/pip-build-tracker-t1gz1yt6
  Created temporary directory: /tmp/pip-install-mh4mlzmc
  Created temporary directory: /tmp/pip-ephem-wheel-cache-xh6xh841
  1 location(s) to search for versions of scikit-build-core:
  * https://pypi.org/simple/scikit-build-core/
  Fetching project page and analyzing links: https://pypi.org/simple/scikit-build-core/
  Getting page https://pypi.org/simple/scikit-build-core/
  Found index url https://pypi.org/simple/
  Looking up "https://pypi.org/simple/scikit-build-core/" in the cache
  Request header has "max_age"

In [4]:

# Импорт необходимых библиотек
import re
import gc
import time
import pickle
import sqlite3
import datetime
import gspread
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.schema import Document
from IPython.display import display, HTML
import os

In [5]:
# Подключение к базе данных SQLite
# from google.colab import drive

import gdown

# Монтируем Google Drive
url = 'https://drive.google.com/drive/folders/1IDo56MQMV6iF4hPD3xzVh6rpZrULRpDp?usp=sharing' # загрузка файла базы данних и файла с креативами
output = '/content/'
gdown.download_folder(url, output=output, quiet=False)

Retrieving folder contents


Processing file 12oHz4ffJey73uVFkfpCdhukUMSq32MOb channels_for_UII.db
Processing file 1rZOppg71FkBND0Ej4XpTIZDXsCfIbpxh Материалы к проекту Silantev studio от заказчика.txt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=12oHz4ffJey73uVFkfpCdhukUMSq32MOb
To: /content/Silantiev/channels_for_UII.db
100%|██████████| 50.6M/50.6M [00:02<00:00, 24.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1rZOppg71FkBND0Ej4XpTIZDXsCfIbpxh
To: /content/Silantiev/Материалы к проекту Silantev studio от заказчика.txt
100%|██████████| 1.44k/1.44k [00:00<00:00, 4.40MB/s]
Download completed


['/content/Silantiev/channels_for_UII.db',
 '/content/Silantiev/Материалы к проекту Silantev studio от заказчика.txt']

Загрузка LLM-модели

In [6]:
# загрузка модели
!wget https://huggingface.co/QuantFactory/Mistral-Nemo-Instruct-2407-GGUF/resolve/main/Mistral-Nemo-Instruct-2407.Q5_K_M.gguf

# Загрузка модели Mistral (GGUF формат)
MODEL_PATH = '/content/Mistral-Nemo-Instruct-2407.Q5_K_M.gguf'

--2024-12-11 14:36:22--  https://huggingface.co/QuantFactory/Mistral-Nemo-Instruct-2407-GGUF/resolve/main/Mistral-Nemo-Instruct-2407.Q5_K_M.gguf
Resolving huggingface.co (huggingface.co)... 13.35.210.114, 13.35.210.61, 13.35.210.66, ...
Connecting to huggingface.co (huggingface.co)|13.35.210.114|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/da/1c/da1cb2bed7636292a31003b0134a031d175c245c21b65b2dbdeffe09d648b267/02783c0f1bcd4946696bae17d6004034fa1a30f1ae4b6e1e7c040293cfe12e13?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Mistral-Nemo-Instruct-2407.Q5_K_M.gguf%3B+filename%3D%22Mistral-Nemo-Instruct-2407.Q5_K_M.gguf%22%3B&Expires=1734186982&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTczNDE4Njk4Mn19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zL2RhLzFjL2RhMWNiMmJlZDc2MzYyOTJhMzEwMDNiMDEzNGEwMzFkMTc1YzI0NWMyMWI2NWIyZGJkZWZmZTA5ZDY0OGIyNjcvMDI3ODNjMGYxYm

Загрузка embedding модели, можно загрузить несколько, для каждой модели будет создана вкладка в google таблице

In [7]:
# Установка путей и констант
DATABASE_PATH = '/content/Silantiev/channels_for_UII.db'  # Путь к базе данных SQLite
EMBEDDINGS_DIR = 'embeddings'          # Директория для сохранения эмбеддингов

SENTENCE_TRANSFORMER_MODELS = [
    "sentence-transformers/distilbert-multilingual-nli-stsb-quora-ranking"  # Используемая модель SentenceTransformer
]

In [8]:

# Подключение к базе данных SQLite
try:
    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()
except sqlite3.Error as e:
    print(f"Ошибка подключения к базе данных: {e}")
    exit(1)


In [9]:
# Определение весов критериев и промпта для Llama.cpp

WEIGHTS = {
    "тематика": 0.4,
    "аудитория": 0.35,
    "качество контента": 0.05,
    "активность": 0.15,
    "соответствие формату": 0.05
}

PROMPT_TEMPLATE = """
Оцени соответствие Telegram-канала креативу по 10-балльной шкале (1 - минимальное, 10 - максимальное).
Учитывай следующие критерии, обязательно указывая оценку по каждому.

Креатив: {creative_texts}

Информация о Telegram-канале:
Название: {channel_name}
Описание: {description}
Подписчики: {subscribers}
Активность: {activity_score}

Последние сообщения: {last_messages}

Твоя задача - оценить канал по каждому критерию и предоставить краткое обоснование. Убедись, что ты выводишь оценку в формате "Критерий: <оценка> <обоснование>".

Оцени канал по следующим критериям:

* **Тематика (1-10):** Насколько тематика канала совпадает с тематикой креатива.
* **Аудитория (1-10):** Насколько целевая аудитория канала совпадает с целевой аудиторией креатива.
* **Качество контента (1-10):** Насколько качественный и интересный контент.
* **Активность (1-10):** Оценка активности уже предоставлена и составляет {activity_score}. Используй эту оценку в своем анализе и укажи ее в ответе.
* **Соответствие формату (1-10):** Насколько формат креатива подходит для канала.

После оценки по каждому критерию, предоставь общий вывод о релевантности канала креативу и итоговую оценку от 1 до 10.

Выдай ответ СТРОГО в следующем формате:

Тематика: <оценка> <обоснование>
Аудитория: <оценка> <обоснование>
Качество контента: <оценка> <обоснование>
Активность: {activity_score} <обоснование>
Соответствие формату: <оценка> <обоснование>
Общий вывод: <текстовый вывод>
Итоговый рейтинг: <число от 1 до 10>

Для ответа используй СТРОГО русский язык. Если по какой-то причине ты не можешь оценить критерий, поставь 0 и объясни причину.
"""

In [10]:
from llama_cpp import Llama
import llama_cpp  # Для вывода информации о системе llama.cpp

def clean_text(text):
    """
    Очищает текст от ссылок, эмодзи, специальных символов и лишних пробелов.

    Аргументы:
    text (str): Исходный текст.

    Возвращает:
    str: Очищенный текст.
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+', '', text)          # Удаляем ссылки
    text = re.sub(r'[^\w\s]', '', text)          # Удаляем эмодзи и специальные символы
    text = ' '.join(text.split())                # Убираем лишние пробелы
    return text

def get_channel_metadata(cursor):
    """
    Извлекает метаданные каналов из базы данных.

    Аргументы:
    cursor (sqlite3.Cursor): Курсор базы данных SQLite.

    Возвращает:
    list: Список словарей с метаданными каналов.
    """
    try:
        cursor.execute('''
            SELECT channel_name, title, description, category, subscribers FROM channels;
        ''')
        metadata = [
            {
                'channel_name': row[0],
                'title': clean_text(row[1]),
                'description': clean_text(row[2]),
                'category': row[3],
                'subscribers': row[4]
            }
            for row in cursor.fetchall()
        ]
        return metadata
    except sqlite3.Error as e:
        print(f"Ошибка при получении метаданных каналов: {e}")
        return []

def get_documents(cursor, metadata):
    """
    Создает документы из сообщений каналов для эмбеддингов и собирает данные активности.

    Аргументы:
    cursor (sqlite3.Cursor): Курсор базы данных SQLite.
    metadata (list): Список метаданных каналов.

    Возвращает:
    list: Список объектов Document с контентом и метаданными.
    """
    documents = []
    for channel in metadata:
        channel_name = channel.get('channel_name', None)
        if not channel_name:
            continue

        try:
            cursor.execute('''
                SELECT text, views_count, shares_count, reactions_count FROM messages
                WHERE channel_name = ?
                ORDER BY date DESC
                LIMIT 20
            ''', (channel_name,))
            messages_for_embedding = cursor.fetchall()

            # Объединяем тексты сообщений для создания эмбеддинга
            embedding_content = ' '.join(
                [clean_text(msg[0].replace('\n', ' ')) for msg in messages_for_embedding if msg[0]]
            )

            doc_metadata = channel.copy()

            # Суммируем показатели активности
            total_views = sum(msg[1] for msg in messages_for_embedding if msg[1] is not None)
            total_shares = sum(msg[2] for msg in messages_for_embedding if msg[2] is not None)
            total_reactions = sum(msg[3] for msg in messages_for_embedding if msg[3] is not None)

            doc_metadata['activity'] = {
                'views': total_views,
                'shares': total_shares,
                'reactions': total_reactions
            }

            documents.append(Document(page_content=embedding_content, metadata=doc_metadata))

        except sqlite3.Error as e:
            print(f"Ошибка при получении сообщений для {channel_name}: {e}")

    return documents

def get_last_three_messages(cursor, metadata):
    """
    Извлекает последние три сообщения из базы данных и добавляет их в метаданные.

    Аргументы:
    cursor (sqlite3.Cursor): Курсор базы данных SQLite.
    metadata (list): Список метаданных каналов.
    """
    try:
        for channel in metadata:
            channel_name = channel.get('channel_name', None)
            if not channel_name:
                continue
            channel['last_messages'] = ''  # Инициализируем поле

            cursor.execute('''
                SELECT text FROM messages
                WHERE channel_name = ?
                ORDER BY date DESC
                LIMIT 3
            ''', (channel_name,))
            messages_3 = cursor.fetchall()

            # Преобразуем последние три сообщения в одну строку
            last_three_messages_str = ' '.join(
                [clean_text(message[0].replace('\n', ' ')) for message in messages_3 if message[0]]
            )

            channel['last_messages'] = last_three_messages_str

    except sqlite3.Error as e:
        print(f"Ошибка при получении последних трех сообщений: {e}")

def create_embedding_database(documents, model_name):
    """
    Создает и сохраняет базу эмбеддингов.

    Аргументы:
    documents (list): Список документов для создания эмбеддингов.
    model_name (str): Название используемой модели.

    Возвращает:
    FAISS: Объект базы эмбеддингов.
    """
    try:
        embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'tokenizer_kwargs': {'use_fast': False}}
        )
        db = FAISS.from_documents(documents, embeddings)
        os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
        embedding_file = os.path.join(EMBEDDINGS_DIR, f"{model_name.replace('/', '_')}_faiss.pkl")
        with open(embedding_file, 'wb') as f:
            pickle.dump(db, f)
        return db
    except Exception as e:
        raise Exception(f"Ошибка при создании базы эмбеддингов: {e}")

def load_embedding_database(model_name):
    """
    Загружает сохранённую базу эмбеддингов.

    Аргументы:
    model_name (str): Название используемой модели.

    Возвращает:
    FAISS или None: Загруженная база эмбеддингов или None, если файл не найден.
    """
    try:
        embedding_file = os.path.join(EMBEDDINGS_DIR, f"{model_name.replace('/', '_')}_faiss.pkl")
        if os.path.exists(embedding_file):
            with open(embedding_file, 'rb') as f:
                db = pickle.load(f)
            return db
        else:
            return None
    except Exception as e:
        raise Exception(f"Ошибка при загрузке базы эмбеддингов: {e}")

def find_similar_channels(creative_texts, db, model_name, k):
    """
    Находит похожие каналы на основе креативных текстов.

    Аргументы:
    creative_texts (list): Список креативных текстов.
    db (FAISS): База эмбеддингов.
    model_name (str): Название используемой модели.
    k (int): Количество возвращаемых каналов.

    Возвращает:
    list: Список кортежей (документ, сходство).
    """
    try:
        combined_creative_text = " ".join(creative_texts)
        cleaned_creative_text = clean_text(combined_creative_text)

        embeddings = HuggingFaceEmbeddings(model_name=model_name)
        creative_embedding = embeddings.embed_query(cleaned_creative_text)

        similarities = []
        unique_channels = set()

        for i, document in enumerate(db.docstore._dict.values()):
            channel_name = document.metadata['channel_name']

            if channel_name not in unique_channels:
                channel_embedding = db.index.reconstruct(i)
                similarity = cosine_similarity([creative_embedding], [channel_embedding])[0][0]
                similarities.append((document, similarity))
                unique_channels.add(channel_name)

        # Сортируем по убыванию сходства
        similarities = sorted(similarities, key=lambda x: x[1], reverse=True)

        # Если найдено меньше каналов, чем k, дополняем списком всех каналов
        if len(similarities) < k:
            all_documents = list(db.docstore._dict.values())
            for document in all_documents:
                channel_name = document.metadata['channel_name']
                if channel_name not in unique_channels:
                    similarities.append((document, 0))
                    unique_channels.add(channel_name)
                if len(similarities) >= k:
                    break

        return similarities[:k]

    except Exception as e:
        raise Exception(f"Ошибка при поиске похожих каналов: {e}")

def get_key_phrases_from_file(filename):
    """
    Извлекает ключевые фразы из текстового файла.

    Аргументы:
    filename (str): Имя файла.

    Возвращает:
    list: Список ключевых фраз.
    """
    with open(filename, 'r', encoding='utf-8') as file:
        content = file.read()
        projects = re.findall(r'Проект №\d+\..*?(?=Проект №|\Z)', content, re.S)
        key_phrases = []
        for project in projects:
            phrases = re.findall(r'(?<=\n)\s*(.*?)(?=<ссылка>)', project)
            key_phrases.extend([phrase.strip() for phrase in phrases if phrase.strip()])
        return key_phrases

def get_activity_score(activity, avg_views, avg_shares, avg_reactions, max_views, max_shares, max_reactions):
    """
    Оценивает активность канала по 10-балльной шкале.

    Аргументы:
    activity (dict): Словарь с данными активности канала.
    avg_views, avg_shares, avg_reactions (float): Средние значения активности по всем каналам.
    max_views, max_shares, max_reactions (float): Максимальные значения активности по всем каналам.

    Возвращает:
    float: Оценка активности канала.
    """
    views = activity.get('views', 0)
    shares = activity.get('shares', 0)
    reactions = activity.get('reactions', 0)

    # Проверка на нулевую активность
    if views == 0 and shares == 0 and reactions == 0:
        return 0.1  # Минимальная оценка

    # Предотвращаем деление на ноль
    max_views = max_views or 1
    max_shares = max_shares or 1
    max_reactions = max_reactions or 1

    # Расчет оценок по каждому показателю
    views_score = min((views / max_views) * 10, 10) if views > 0 else 0
    shares_score = min((shares / max_shares) * 10, 10) if shares > 0 else 0
    reactions_score = min((reactions / max_reactions) * 10, 10) if reactions > 0 else 0

    total_score = (views_score + shares_score + reactions_score) / 3
    total_score_rounded = round(total_score, 2)  # Округляем до двух знаков после запятой

    # Вывод подробной информации для отладки
    print(f"Расчет активности для канала {activity.get('channel_name', 'N/A')}:")
    print(f"  Просмотры: {views}, Оценка: {views_score}")
    print(f"  Репосты: {shares}, Оценка: {shares_score}")
    print(f"  Реакции: {reactions}, Оценка: {reactions_score}")
    print(f"  Средние просмотры: {avg_views}, Средние репосты: {avg_shares}, Средние реакции: {avg_reactions}")
    print(f"  Максимальные просмотры: {max_views}, Максимальные репосты: {max_shares}, Максимальные реакции: {max_reactions}")
    print(f"  Итоговая оценка активности: {total_score_rounded}")

    return total_score_rounded

def compute_activity_stats(documents, cursor):
    """
    Вычисляет статистику активности по каналам.

    Аргументы:
    documents (list): Список документов каналов.
    cursor (sqlite3.Cursor): Курсор базы данных SQLite.

    Возвращает:
    tuple: Средние и максимальные значения просмотров, репостов и реакций.
    """
    total_views_list = []
    total_shares_list = []
    total_reactions_list = []

    for doc in documents:
        channel_name = doc.metadata.get('channel_name', '')
        try:
            cursor.execute('''
                SELECT SUM(views_count), SUM(shares_count), SUM(reactions_count) FROM messages
                WHERE channel_name = ?
                ORDER BY date DESC
                LIMIT 20
            ''', (channel_name,))
            activity_data = cursor.fetchone()
            views = activity_data[0] or 0
            shares = activity_data[1] or 0
            reactions = activity_data[2] or 0

            if views > 0:
                total_views_list.append(views)
            if shares > 0:
                total_shares_list.append(shares)
            if reactions > 0:
                total_reactions_list.append(reactions)

        except sqlite3.Error as e:
            print(f"Ошибка при получении данных активности для {channel_name}: {e}")

    max_views = max(total_views_list) if total_views_list else 1
    max_shares = max(total_shares_list) if total_shares_list else 1
    max_reactions = max(total_reactions_list) if total_reactions_list else 1

    avg_views = sum(total_views_list) / len(total_views_list) if total_views_list else 0
    avg_shares = sum(total_shares_list) / len(total_shares_list) if total_shares_list else 0
    avg_reactions = sum(total_reactions_list) / len(total_reactions_list) if total_reactions_list else 0

    # Вывод статистики активности
    print("Статистика активности по каналам с ненулевой активностью:")
    print(f"  Обработано каналов с активностью: {len(total_views_list)}")
    print(f"  Максимальные просмотры: {max_views}, Средние просмотры: {avg_views}")
    print(f"  Максимальные репосты: {max_shares}, Средние репосты: {avg_shares}")
    print(f"  Максимальные реакции: {max_reactions}, Средние реакции: {avg_reactions}")

    return avg_views, avg_shares, avg_reactions, max_views, max_shares, max_reactions

def rank_channels_llama_cpp(creative_texts, channel_info, llm, avg_views, avg_shares, avg_reactions, max_views, max_shares, max_reactions, cursor):
    """
    Ранжирует каналы с помощью модели Llama.cpp на основе различных критериев.

    При проставлении оценки активности и расчете итоговой оценки используется "Итоговая оценка активности: {total_score_rounded}" из функции get_activity_score.

    Аргументы:
    creative_texts (list): Список креативных текстов.
    channel_info (tuple): Кортеж (документ, сходство).
    llm (Llama): Загруженная модель Llama.cpp.
    avg_views, avg_shares, avg_reactions (float): Средние значения активности.
    max_views, max_shares, max_reactions (float): Максимальные значения активности.
    cursor (sqlite3.Cursor): Курсор базы данных SQLite.

    Возвращает:
    dict: Словарь с результатами ранжирования и дополнительной информацией.
    """
    document, score = channel_info
    metadata = document.metadata
    channel_name = metadata['channel_name']
    title = metadata['title']
    category = metadata['category']
    description = metadata['description']
    subscribers = metadata.get('subscribers', 'N/A')
    last_messages = metadata.get('last_messages', '')

    try:
        cursor.execute('''
            SELECT SUM(views_count), SUM(shares_count), SUM(reactions_count) FROM messages
            WHERE channel_name = ?
            ORDER BY date DESC
            LIMIT 20
        ''', (channel_name,))
        activity_data = cursor.fetchone()
        activity = {
            'views': activity_data[0] or 0,
            'shares': activity_data[1] or 0,
            'reactions': activity_data[2] or 0,
            'channel_name': channel_name
        }

    except sqlite3.Error as e:
        print(f"Ошибка при получении данных активности для {channel_name}: {e}")
        activity = {'views': 0, 'shares': 0, 'reactions': 0, 'channel_name': channel_name}

    # Получаем оценку активности из функции get_activity_score
    activity_score = get_activity_score(activity, avg_views, avg_shares, avg_reactions, max_views, max_shares, max_reactions)

    prompt = PROMPT_TEMPLATE.format(
        creative_texts=' '.join(creative_texts),
        channel_name=channel_name,
        description=description,
        subscribers=subscribers,
        activity_score=activity_score,
        last_messages=last_messages
    )

    output = llm(prompt=prompt, max_tokens=512, temperature=0.3)
    ranking_text = output['choices'][0]['text'].strip()

    ratings = {}
    ratings['активность'] = activity_score

    for criterion in WEIGHTS:
        if criterion == 'активность':
            continue
        match = re.search(rf"{criterion}:\s*(\d+(?:\.\d+)?)", ranking_text, re.IGNORECASE | re.MULTILINE)

        if match:
            try:
                ratings[criterion] = float(match.group(1))
            except ValueError:
                print(f"Ошибка: некорректное значение для {criterion}: {match.group(1)}")
                ratings[criterion] = 0
        else:
            print(f"Ошибка: не найдено значение для {criterion}")
            ratings[criterion] = 0

    # **Изменение**: Округляем промежуточные расчеты и итоговый рейтинг до двух знаков после запятой
    rating = round(sum(round(ratings.get(criterion, 0) * WEIGHTS[criterion], 2) for criterion in WEIGHTS), 2)
    comment = ranking_text.split("Итоговый рейтинг:")[0].strip()

    comment += "\n\nРасчет итогового рейтинга:"
    for criterion, weight in WEIGHTS.items():
        individual_score = round(ratings.get(criterion, 0) * weight, 2)
        comment += f"\n{criterion}: {ratings.get(criterion, 0)} * {weight} = {individual_score}"
    comment += f"\nИтого: {rating}"

    return {
        "channel_name": channel_name,
        "title": title,
        "category": category,
        "description": description,
        "last_messages": last_messages,
        "rating": rating,
        "comment": comment,
        "prompt": prompt,
        "score": score
    }


In [11]:
# Основная функция

def main():
    """
    Основная функция выполнения скрипта.

    Шаги:
    1. Извлечение метаданных каналов и создание документов.
    2. Загрузка или создание базы эмбеддингов.
    3. Поиск похожих каналов для каждого креативного текста.
    4. Ранжирование каналов с помощью модели Llama.cpp.
    5. Сохранение результатов в Google Sheets.
    """
    # Извлечение метаданных каналов и создание документов
    metadata = get_channel_metadata(cursor)
    documents = get_documents(cursor, metadata)
    model_names = SENTENCE_TRANSFORMER_MODELS


    # Извлечение ключевых фраз из файла
    key_phrases = get_key_phrases_from_file('/content/Silantiev/Материалы к проекту Silantev studio от заказчика.txt')

    # Создание Google Sheets для результатов

    spreadsheet = client.create('Results_with_LLaMA_CPP')
    spreadsheet.share(None, perm_type='anyone', role='writer')
    print(f'Ссылка на созданные Google-таблицы: https://docs.google.com/spreadsheets/d/{spreadsheet.id}')

    # Отображение кнопки для открытия Google-таблицы
    spreadsheet_url = f'https://docs.google.com/spreadsheets/d/{spreadsheet.id}'
    display(HTML(f'''
        <button onclick="window.open('{spreadsheet_url}', '_blank')">Открыть созданную Google-таблицу</button>
    '''))



    # Загрузка модели Llama.cpp
    model_path = '/content/Mistral-Nemo-Instruct-2407.Q5_K_M.gguf'
    if not os.path.exists(model_path):
        print(f"Файл модели не найден по пути: {model_path}")
        return


    try:
        n_gpu_layers = 150  # Количество слоев на GPU
        system_info = llama_cpp.llama_cpp.llama_print_system_info().decode('utf-8')
        print("Информация о системе Llama.cpp:")
        print(system_info)

        llm = Llama(
            model_path=model_path,
            n_gpu_layers=n_gpu_layers,
            n_batch=8192,
            n_ctx=8192,
            n_threads=8,
            use_mlock=True,
            f16_kv=True,
            verbose=True,
        )

    except Exception as e:
        print(f"Ошибка при загрузке модели: {e}")
        return

    for model_name in model_names:
        worksheet_title = model_name[:31]  # Ограничение на длину названия листа
        existing_titles = [sheet.title for sheet in spreadsheet.worksheets()]
        if worksheet_title in existing_titles:
            worksheet_title += f'_{datetime.datetime.now().strftime("%Y%m%d%H%M%S")}'
        worksheet = spreadsheet.add_worksheet(title=worksheet_title, rows="1000", cols="20")
        try:
            default_sheet = spreadsheet.get_worksheet(0)
            if default_sheet and (default_sheet.title == 'Лист1' or default_sheet.title == 'Sheet1'):
                spreadsheet.del_worksheet(default_sheet)
        except gspread.exceptions.WorksheetNotFound:
            pass  # Лист уже удалён
        worksheet.freeze(1)
        worksheet.append_row([
            'model', 'channel', 'title', 'category', 'description', 'score',
            'last_messages', 'key_phrase', 'tester_comment', 'rating from -2 to 2',
            'LLM_rating', 'LLM_comment'
        ])

        for creative_text in key_phrases:
            db = load_embedding_database(model_name)
            if db is None:
                db = create_embedding_database(documents, model_name)

            similar_channels = find_similar_channels([creative_text], db, model_name, k=20)
            print(f"Модель: {model_name}, ключевая фраза: '{creative_text}', найдено каналов: {len(similar_channels)}")

            # Извлекаем метаданные только для найденных каналов и обновляем last_messages
            similar_documents_metadata = [doc.metadata for doc, _ in similar_channels]
            get_last_three_messages(cursor, similar_documents_metadata)

            similar_documents = [channel_data[0] for channel_data in similar_channels]
            avg_views, avg_shares, avg_reactions, max_views, max_shares, max_reactions = compute_activity_stats(similar_documents, cursor)

            ranking_results = []
            for channel_data in similar_channels:
                ranking_result = rank_channels_llama_cpp(
                    [creative_text], channel_data, llm,
                    avg_views, avg_shares, avg_reactions,
                    max_views, max_shares, max_reactions, cursor
                )
                ranking_results.append(ranking_result)

            # Сортируем результаты по убыванию рейтинга
            ranking_results.sort(key=lambda x: x['rating'], reverse=True)

            # Сортируем результаты по рейтингу
            for ranking_result in ranking_results[:5]:
                try:
                    print(f"Запись в Google Sheets: {ranking_result}")
                    # Преобразование рейтинга в числовой формат
                    ranking_result['rating'] = float(ranking_result['rating'])
                    worksheet.append_row([
                        model_name,
                        ranking_result['channel_name'],
                        ranking_result['title'],
                        ranking_result['category'],
                        ranking_result['description'],
                        ranking_result.get('score',''),
                        ranking_result['last_messages'],
                        creative_text,
                        '',  # tester_comment
                        '',  # rating from -2 to 2
                        ranking_result['rating'],
                        ranking_result['comment']
                    ])
                    print("Данные успешно записаны.")
                    time.sleep(1)  # Пауза для предотвращения превышения лимитов API
                except Exception as e:
                    print(f"Ошибка при записи в Google Sheets: {e}")

            del db
            gc.collect()

    conn.close()


Запуск работы кода. Ссылку на созданную Google таблицу ищите в начале лога!!

In [12]:
main()

Ссылка на созданные Google-таблицы: https://docs.google.com/spreadsheets/d/1gKs70uazVIrQX-RL9hYpMV4qYsV3xsCMd2k9fABIsak


ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes


Информация о системе Llama.cpp:
CUDA : ARCHS = 750 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | LLAMAFILE = 1 | OPENMP = 1 | AARCH64_REPACK = 1 | 


llama_load_model_from_file: using device CUDA0 (Tesla T4) - 14999 MiB free
llama_model_loader: loaded meta data with 32 key-value pairs and 363 tensors from /content/Mistral-Nemo-Instruct-2407.Q5_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Models
llama_model_loader: - kv   3:                         general.size_label str              = 12B
llama_model_loader: - kv   4:                            general.license str              = apache-2.0
llama_model_loader: - kv   5:                          general.languages arr[str,9]       = ["en", "fr", "de", "es", "it", "pt", ...
llama_model_loader: - kv   6:              

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

Расчет активности для канала @travelac:
  Просмотры: 148097, Оценка: 0.09656290388855801
  Репосты: 1102, Оценка: 0.19412344983089064
  Реакции: 1284, Оценка: 0.18586876275676378
  Средние просмотры: 1296133.0, Средние репосты: 5180.7, Средние реакции: 7894.8
  Максимальные просмотры: 15336842, Максимальные репосты: 56768, Максимальные реакции: 69081
  Итоговая оценка активности: 0.16


llama_perf_context_print:        load time =    1887.68 ms
llama_perf_context_print: prompt eval time =       0.00 ms /  1960 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /   351 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =   25629.97 ms /  2311 tokens
Llama.generate: 123 prefix-match hit, remaining 851 prompt tokens to eval


Расчет активности для канала @bankvtb:
  Просмотры: 15336842, Оценка: 10.0
  Репосты: 13141, Оценка: 2.3148604847801577
  Реакции: 28636, Оценка: 4.145278730765333
  Средние просмотры: 1296133.0, Средние репосты: 5180.7, Средние реакции: 7894.8
  Максимальные просмотры: 15336842, Максимальные репосты: 56768, Максимальные реакции: 69081
  Итоговая оценка активности: 5.49


llama_perf_context_print:        load time =    1887.68 ms
llama_perf_context_print: prompt eval time =       0.00 ms /   851 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =    1481.31 ms /   852 tokens


Ошибка: не найдено значение для тематика
Ошибка: не найдено значение для аудитория
Ошибка: не найдено значение для качество контента
Ошибка: не найдено значение для соответствие формату
Запись в Google Sheets: {'channel_name': '@crypto_system_channell', 'title': 'Crypto System Trading', 'category': 'Криптовалюты', 'description': 'связь с редакцией Ghttechbot', 'last_messages': 'Santiment Ethereum самый обсуждаемый толпой криптоактив в соц сетях Внимание приковано к решению SEC одобрить первый спотовый ETHETF Variant Fund CLO Джейк Червински Если спотовый ETHETF будет одобрен это станет настоящим шоком для всех Одобрение может означать серьезный сдвиг в криптовалютной политике США QCP Capital ETH может вырасти до 5000 если спотовый ETHETF будет одобрен Дедлайны ETHETF обновлены Fidelity обновила заявку S1 на спотовый ETHETF', 'rating': 8.46, 'comment': '---\n\nТематика: 10. Канал посвящен криптовалюте, что полностью соответствует тематике креатива.\nАудитория: 10. Целевая аудитория кана